# CEWAY Colab batch runner

Runs deterministic DLT/SSQ historical backtests and stores JSON outputs in Google Drive. Production draw updates, the web app, and recommendation delivery remain outside Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AI-Projects/ceway')
for name in ('datasets', 'models', 'outputs', 'checkpoints'):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/Wisely88/Ceway-Digital-Decision-Platform.git'
REPO_DIR = Path('/content/Ceway-Digital-Decision-Platform')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'backend/requirements.txt'], check=True)

In [ ]:
# Validate the backend before producing research outputs.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'backend/tests', '-v'], check=True)

In [ ]:
from datetime import datetime, timezone

GAMES = ('dlt', 'ssq')
BUDGET = 20
PERIODS = 100
WINDOW = 100
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
for game in GAMES:
    output = DRIVE_ROOT / 'outputs' / f'{stamp}-{game}-backtest.json'
    subprocess.run([sys.executable, 'scripts/run_colab_batch.py', '--game', game, '--budget', str(BUDGET), '--periods', str(PERIODS), '--window', str(WINDOW), '--output', str(output)], check=True)
print('Backtests completed:', DRIVE_ROOT / 'outputs')